In [35]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, KFold
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import optuna
import shap
import pickle
import joblib
import matplotlib.pyplot as plt

import gc
import sys
import os
from pathlib import Path



#Indicamos el workspace principal (dos niveles atrás)
sys.path.append(os.path.abspath('../../'))

In [10]:

from src.utils_pisa import *


%load_ext autoreload
%autoreload 2


ruta_datos_procesados = Path('../../data/processed')

establecer_semilla(42)
pd.set_option('display.float_format', lambda valor: f'{valor:.2f}')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Semillas del proyecto establecidas con éxito (seed=42).


# Modelado Explicativo e Inferencia (Tabla Larga)

## Generación datos train & test

In [ ]:
#Carga de datos pre procesados para el modelo
df_modelo_estrato = pd.read_parquet(
    ruta_datos_procesados / 'datos_modelo_estrato.parquet'
)

In [12]:
print(list(df_modelo_estrato.columns))

['year', 'media_math_estrato', 'media_read_estrato', 'media_scie_estrato', 'tiempo_deberes_totales_diarios', 'dispone_personal_apoyo', 'PIB_per_capita_PPP', 'Gasto_Alumno_PPP', 'Gasto_Educacion_PIB', 'Indice_Gini', 'ESCS_media_pais_anual', 'Presion_Academica_Neta', 'ECS_categoria_Bajo', 'ECS_categoria_Medio', 'Perfil_Pais_Alto Esfuerzo Educ_Bajo ESCS', 'Perfil_Pais_Emergente_Alta Desigualdad', 'Perfil_Pais_Modelo Igualitario_Inversión Mod.']


In [17]:
#Separamos las variables para poder realizar los tres modelos

#Covariables
X = df_modelo_estrato.drop(columns=['media_math_estrato','media_scie_estrato','media_read_estrato'])

#Targets
y = df_modelo_estrato[['media_math_estrato', 'media_scie_estrato', 'media_read_estrato']]

Dado que para cada año el numero de datos cambiaba en función del numero de estratos disponibles, nos aseguramos de mantener la misma proporcion 80/20 indicando a la función de separación la estratificación por la columna de año

In [19]:
#Separamos los datos en train & test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    stratify=X['year'], #Estratificación por años
    test_size=0.2,      #Separacion 80/20
)

y_train_mat = y_train['media_math_estrato'] 
y_train_sci = y_train['media_scie_estrato']
y_train_rd =  y_train['media_read_estrato']

y_test_mat = y_test['media_math_estrato'] 
y_test_sci = y_test['media_scie_estrato']
y_test_read  = y_test['media_read_estrato']


## Entrenamiento de modelo

### Modelo para matemáticas

In [ ]:
#Creacion del modelo de regresion
xgb_math = xgb.XGBRegressor(
    n_estimators=100, 
    learning_rate=0.1, 
)
xgb_math.fit(X_train, y_train_mat)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [41]:

y_pred_mat = xgb_math.predict(X_test)

r2 = r2_score(y_test_mat, y_pred_mat)
mae = mean_absolute_error(y_test_mat, y_pred_mat)
rmse = np.sqrt(mean_squared_error(y_test_mat, y_pred_mat))

print("--- Desempeño XGBoost (Matemáticas)(BASE) ---")
print(f"R2: {r2:.2f} (Varianza explicada)")
print(f"MAE: {mae:.2f} puntos (Error promedio absoluto)")
print(f"RMSE: {rmse:.2f} puntos")

--- Desempeño XGBoost (Matemáticas)(BASE) ---
R2: 0.40 (Varianza explicada)
MAE: 36.99 puntos (Error promedio absoluto)
RMSE: 51.23 puntos


#### Afinación de hiperparámetros (optuna)

In [36]:

#Función objetivo que optuna optimizara
def objective(trial):
    # Definimos los rangos
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
    }

    #Definimos el modelo con los parámetros 
    modelo = xgb.XGBRegressor(**param)
    
    # Calculamos el error RMSE usando validación cruzada
    scores = cross_val_score(
        modelo, 
        X_train, 
        y_train_mat, 
        scoring='neg_root_mean_squared_error', 
        cv=3
    )

    # Convertimos el score negativo a positivo (RMSE real) para que Optuna lo minimice
    rmse = -1 * scores.mean()
    return rmse

In [37]:
#Creamos el estudio de optuna
study = optuna.create_study(direction='minimize')

# Ejecutamos la búsqueda (n_trials es el número de combinaciones a probar)
study.optimize(objective, n_trials=20) 



[I 2026-10-07 12:40:35,837] A new study created in memory with name: no-name-f898694b-8be3-4872-979d-d25a4b5a033a
[I 2026-10-07 12:40:36,593] Trial 0 finished with value: 48.647212982177734 and parameters: {'n_estimators': 117, 'learning_rate': 0.06013665498787748, 'max_depth': 6, 'subsample': 0.5312630289687146}. Best is trial 0 with value: 48.647212982177734.
[I 2026-10-07 12:40:38,360] Trial 1 finished with value: 54.612405141194664 and parameters: {'n_estimators': 379, 'learning_rate': 0.2431588012112856, 'max_depth': 5, 'subsample': 0.5658346826647789}. Best is trial 0 with value: 48.647212982177734.
[I 2026-10-07 12:40:38,716] Trial 2 finished with value: 52.84204610188802 and parameters: {'n_estimators': 104, 'learning_rate': 0.023538466650375345, 'max_depth': 3, 'subsample': 0.7705064819391028}. Best is trial 0 with value: 48.647212982177734.
[I 2026-10-07 12:40:40,127] Trial 3 finished with value: 49.70138676961263 and parameters: {'n_estimators': 125, 'learning_rate': 0.05463

In [38]:
# Entrenar el modelo final con los mejores parámetros encontrados
print("Mejores parámetros encontrados:", study.best_params)
xgb_math_optimizado = xgb.XGBRegressor(**study.best_params, random_state=42)
xgb_math_optimizado.fit(X_train, y_train_mat)

Mejores parámetros encontrados: {'n_estimators': 191, 'learning_rate': 0.03259605676603692, 'max_depth': 6, 'subsample': 0.6438195273250038}


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03259605676603692, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=191, n_jobs=None,
             num_parallel_tree=None, ...)

In [39]:
y_pred_mat = xgb_math_optimizado.predict(X_test)

r2_opt = r2_score(y_test_mat, y_pred_mat)
mae_opt = mean_absolute_error(y_test_mat, y_pred_mat)
rmse_opt = np.sqrt(mean_squared_error(y_test_mat, y_pred_mat))

print("--- Desempeño XGBoost (Matemáticas)(Optimizado) ---")

print(f"R2 Optimizado: {r2_opt:.4f}")
print(f"MAE Optimizado: {mae_opt:.2f} puntos ")
print(f"RMSE Optimizado: {rmse_opt:.2f}")



--- Desempeño XGBoost (Matemáticas)(Optimizado) ---
R2 Optimizado: 0.4219
MAE Optimizado: 36.26 puntos 
RMSE Optimizado: 50.24


In [40]:
# 2. Crear la tabla consolidando las métricas
tabla_comparativa = pd.DataFrame({
    'XGBoost Base': [
        r2,
        mae,
        rmse
    ],
    'XGBoost Optimizado': [
        r2_opt,
        mae_opt,
        rmse_opt
    ]
}, index=['R2 (↑ Mejor)', 'MAE (↓ Mejor)', 'RMSE (↓ Mejor)'])

tabla_comparativa.round(2)

,XGBoost Base,XGBoost Optimizado
R2 (↑ Mejor),0.40,0.42
MAE (↓ Mejor),36.99,36.26
RMSE (↓ Mejor),51.23,50.24


# Modelado Predictivo y Pronósticos (Tabla Ancha) TODO

# Exportacion de datos y modelos

In [45]:
ruta_modelos = Path('../../data/models')
ruta_datos_llm = Path('../../agente_llm/data_llm')

In [ ]:
#Guardamos los datos empleados para test
joblib.dump(X_test, ruta_modelos / 'X_test_preparado.pkl')
joblib.dump(xgb_math_optimizado, ruta_modelos/'modelo_xgb_math_opt.pkl')


#Almacenamiendo de datos para LLM
joblib.dump(xgb_math_optimizado, ruta_datos_llm/'modelo_xgb_math_opt.pkl')

columnas_modelo = X_train.columns.tolist()
joblib.dump(columnas_modelo, ruta_datos_llm / 'columnas_modelo.pkl')


['..\\..\\agente_llm\\data_llm\\columnas_modelo.pkl']